## **Background**

Retention rate is one of the customer behaviors that can be necessary things in evaluating business performance. Retention rate shows the measurement of customer loyalty to consume our product in certain period. This project will analyze retention rate from one kind of retail store using cohort analysis.

Data Overview:
1. order_id : an unique code from every transaction.
2. product_code : an unique code from product
3. product_name : name of a product
4. quantity : number of goods ordered
5. order_date : datetime order created
6. price : price of a product
7. customer_id : an unique code from customer

In [6]:
import pandas as pd
import numpy as np 
import datetime as datetime
import seaborn as sns
import matplotlib.pyplot as plt
import re
import datetime as dt
from scipy import stats

In [7]:
# Import dataset
file= 'https://drive.google.com/file/d/1ELg9NNsaC44Q8r-Y7i8UKzUwZuNKFEom/view?usp=sharing'
url= 'https://drive.google.com/uc?id=' + file.split('/')[-2]
datar= pd.read_csv(url)
datar.head()

,order_id,product_code,product_name,quantity,order_date,price,customer_id
0,493410,TEST001,This is a test product.,5,2010-01-04 09:24:00,4.50,12346.0
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1,2010-01-04 09:43:00,4.25,14590.0
2,493412,TEST001,This is a test product.,5,2010-01-04 09:53:00,4.50,12346.0
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1,2010-01-04 09:54:00,0.85,NaN
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1,2010-01-04 09:54:00,3.75,NaN


## **Data Understanding**

In [8]:
# Dataframe info
data= datar.copy()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 461773 entries, 0 to 461772
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      461773 non-null  object 
 1   product_code  461773 non-null  object 
 2   product_name  459055 non-null  object 
 3   quantity      461773 non-null  int64  
 4   order_date    461773 non-null  object 
 5   price         461773 non-null  float64
 6   customer_id   360853 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 24.7+ MB


In [9]:
data.isnull().sum()

order_id             0
product_code         0
product_name      2718
quantity             0
order_date           0
price                0
customer_id     100920
dtype: int64

In [28]:
# Order ID check

pop=[]
for i in data['order_id']:
    op=re.findall(r'\D',i)
    if op not in pop and op!=[]:
        pop.append(op)
print(pop)


[['C'], ['A']]


In [11]:
# Whats 'C' in order_id
(data[data['order_id'].str.contains('C')]).head()

,order_id,product_code,product_name,quantity,order_date,price,customer_id
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1,2010-01-04 09:43:00,4.25,14590.0
12,C493415,21527,RETRO SPOT TRADITIONAL TEAPOT,-3,2010-01-04 10:33:00,7.95,14590.0
13,C493426,22109,FULL ENGLISH BREAKFAST PLATE,-1,2010-01-04 10:41:00,3.39,16550.0
56,C493430,21556,CERAMIC STRAWBERRY MONEY BOX,-1,2010-01-04 11:43:00,2.55,14680.0
57,C493430,21232,STRAWBERRY CERAMIC TRINKET BOX,-2,2010-01-04 11:43:00,1.25,14680.0


In [29]:
# Product Code Check

pup=[]
for i in data['product_code']:
    op=re.findall(r'^([A-Za-z])[A-Za-z]*\d*\D*+$',i)
    if i not in pup and op!=[]:
        pup.append(i)
print(pup)


['TEST001', 'POST', 'M', 'DOT', 'DCGS0058', 'BANK CHARGES', 'D', 'PADS', 'DCGS0068', 'DCGS0076', 'ADJUST', 'DCGSSGIRL', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0039', 'DCGS0060', 'DCGS0056', 'DCGS0059', 'GIFT', 'DCGSLBOY', 'C2', 'm', 'DCGS0053', 'DCGS0004', 'DCGS0062', 'DCGS0037', 'DCGSSBOY', 'DCGSLGIRL', 'S', 'DCGS0069', 'DCGS0070', 'DCGS0075', 'B', 'DCGS0041', 'DCGS0003', 'ADJUST2', 'C3', 'SP1002', 'AMAZONFEE']


In [39]:
# Product_code check
(data[data['product_code']=='DCGS0027'])

,order_id,product_code,product_name,quantity,order_date,price,customer_id
32412,496742,DCGS0027,NaN,-1,2010-02-03 14:29:00,0.0,NaN


In [13]:
(data[data['product_name'].str.lower().str.contains('adjust',na=False)]).head()

,order_id,product_code,product_name,quantity,order_date,price,customer_id
23816,C495737,ADJUST,Adjustment by john on 26/01/2010 16,-1,2010-01-26 16:23:00,10.50,16154.0
23817,C495740,ADJUST,Adjustment by john on 26/01/2010 16,-1,2010-01-26 16:24:00,14.00,13054.0
23818,C495739,ADJUST,Adjustment by john on 26/01/2010 16,-1,2010-01-26 16:24:00,10.50,15383.0
23819,C495741,ADJUST,Adjustment by john on 26/01/2010 16,-1,2010-01-26 16:25:00,13.14,16840.0
23863,C495751,ADJUST,Adjustment by john on 26/01/2010 16,-1,2010-01-26 16:28:00,26.19,12858.0


In [14]:
# Quantity Check
data[data['quantity']<0]

,order_id,product_code,product_name,quantity,order_date,price,customer_id
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1,2010-01-04 09:43:00,4.25,14590.0
12,C493415,21527,RETRO SPOT TRADITIONAL TEAPOT,-3,2010-01-04 10:33:00,7.95,14590.0
13,C493426,22109,FULL ENGLISH BREAKFAST PLATE,-1,2010-01-04 10:41:00,3.39,16550.0
56,C493430,21556,CERAMIC STRAWBERRY MONEY BOX,-1,2010-01-04 11:43:00,2.55,14680.0
57,C493430,21232,STRAWBERRY CERAMIC TRINKET BOX,-2,2010-01-04 11:43:00,1.25,14680.0
...,...,...,...,...,...,...,...
460995,C539950,85099B,JUMBO BAG RED RETROSPOT,-10,2010-12-23 11:50:00,1.95,13534.0
460996,C539950,22720,SET OF 3 CAKE TINS PANTRY DESIGN,-2,2010-12-23 11:50:00,4.95,13534.0
461068,C539956,35004C,SET OF 3 COLOURED FLYING DUCKS,-15,2010-12-23 12:55:00,4.65,12980.0
461596,539980,35001W,NaN,-36,2010-12-23 14:34:00,0.00,NaN


In [15]:
# Price Check
data[data['price']<0]

,order_id,product_code,product_name,quantity,order_date,price,customer_id
124462,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN
213524,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN
329774,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN


**NOTE**
1. There is missing value from product_name and customer_id columns.
2. There are some inappropriate data type.
3. There are unique codes from order_id colomns. order_id starts with 'C' will be clasifeid as canceled order and 'A' will be clasdified as Adjustment.
4. There are some unnecessary product code and we assume there aren't transsaction goods from cust, so will be drop soon.
5. Most of a negative values from quantity columns are canceled order or Missing value from product_name.
6. Negative values from price are adjustment product.

## **Data Cleansing**

In [46]:
# Drop All missing value
datac=data.copy()
datac= datac.dropna()
datac.info()

<class 'pandas.core.frame.DataFrame'>
Index: 360853 entries, 0 to 461744
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      360853 non-null  object 
 1   product_code  360853 non-null  object 
 2   product_name  360853 non-null  object 
 3   quantity      360853 non-null  int64  
 4   order_date    360853 non-null  object 
 5   price         360853 non-null  float64
 6   customer_id   360853 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 22.0+ MB


In [47]:
# Check Again Unnecessary value

puc=[]
for i in datac['product_code']:
    op=re.findall(r'^([A-Za-z])[A-Za-z]*\d*\D*+$',i)
    if i not in puc and op!=[]:
        puc.append(i)
print(puc)


['TEST001', 'POST', 'M', 'D', 'PADS', 'ADJUST', 'C2', 'BANK CHARGES', 'ADJUST2', 'SP1002']


In [ ]:
# Drop All Unnecesssary values
seperate='|'
pilter= seperate.join(puc)
pilter

datac= datac[~(datac['product_name'].str.lower().str.contains('adjust|test',case=False) |
             datac['order_id'].str.lower().str.contains('c|a', case=False))]
datac

,order_id,product_code,product_name,quantity,order_date,price,customer_id
6,493414,21844,RETRO SPOT MUG,36,2010-01-04 10:28:00,2.55,14590.0
7,493414,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-04 10:28:00,4.25,14590.0
8,493414,37508,NEW ENGLAND CERAMIC CAKE SERVER,2,2010-01-04 10:28:00,2.55,14590.0
9,493414,35001G,HAND OPEN SHAPE GOLD,2,2010-01-04 10:28:00,4.25,14590.0
10,493414,21527,RETRO SPOT TRADITIONAL TEAPOT,12,2010-01-04 10:28:00,6.95,14590.0
...,...,...,...,...,...,...,...
461740,539988,84380,SET OF 3 BUTTERFLY COOKIE CUTTERS,1,2010-12-23 16:06:00,1.25,18116.0
461741,539988,84849D,HOT BATHS SOAP HOLDER,1,2010-12-23 16:06:00,1.69,18116.0
461742,539988,84849B,FAIRY SOAP SOAP HOLDER,1,2010-12-23 16:06:00,1.69,18116.0
461743,539988,22854,CREAM SWEETHEART EGG HOLDER,2,2010-12-23 16:06:00,4.95,18116.0


In [43]:
# Change data type
datac['order_date']= pd.to_datetime(datac['order_date'])
datac['customer_id']= datac['customer_id'].astype(str)

datac.head()

,order_id,product_code,product_name,quantity,order_date,price,customer_id
6,493414,21844,RETRO SPOT MUG,36,2010-01-04 10:28:00,2.55,14590.0
7,493414,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-04 10:28:00,4.25,14590.0
8,493414,37508,NEW ENGLAND CERAMIC CAKE SERVER,2,2010-01-04 10:28:00,2.55,14590.0
9,493414,35001G,HAND OPEN SHAPE GOLD,2,2010-01-04 10:28:00,4.25,14590.0
10,493414,21527,RETRO SPOT TRADITIONAL TEAPOT,12,2010-01-04 10:28:00,6.95,14590.0


In [44]:
# Check
datac.count()

order_id        352893
product_code    352893
product_name    352893
quantity        352893
order_date      352893
price           352893
customer_id     352893
dtype: int64

In [20]:
# Outlier handling
score= stats.zscore(datac[['quantity','price']])
score.max()

np.float64(350.60590158821554)

In [21]:
parjo= datac[(score>100).any(axis=1)]
parjo

,order_id,product_code,product_name,quantity,order_date,price,customer_id
26972,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949.0
83288,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918.0
83302,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0
83303,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0
83304,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0
83305,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0
260032,521315,17003,BROCADE RING PURSE,7128,2010-09-03 12:01:00,0.19,15838.0
288739,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063.0
288863,524174,21096,SET/6 FRUIT SALAD PAPER PLATES,7008,2010-09-27 16:30:00,0.13,13687.0
288864,524174,21088,SET/6 FRUIT SALAD PAPER CUPS,7128,2010-09-27 16:30:00,0.08,13687.0


In [27]:
data[data['product_code']=='SP1002']

,order_id,product_code,product_name,quantity,order_date,price,customer_id
305328,525772,SP1002,KID'S CHALKBOARD/EASEL,1,2010-10-07 11:12:00,2.95,12748.0
305648,525837,SP1002,KID'S CHALKBOARD/EASEL,4,2010-10-07 12:23:00,2.95,17841.0
350878,530135,SP1002,NaN,-27,2010-11-01 15:33:00,0.00,NaN


In [24]:
data[(data['customer_id']==12931.0)&(data['order_date']=='2010-01-19 16:45:00')]

,order_id,product_code,product_name,quantity,order_date,price,customer_id
15854,C494909,D,Discount,-30,2010-01-19 16:45:00,0.40,12931.0
15855,C494909,D,Discount,-30,2010-01-19 16:45:00,0.13,12931.0


## Coba

In [25]:
s= {'Tanggal':['2024-09-04','2024-03-07',202311],
    'Data':['A','B',None]}
afk= pd.DataFrame(s)
afk['Tanggal']= pd.to_datetime(afk['Tanggal'], format='%Y%m%d')
# afk['Tanggal']= afk['Tanggal'].astype('datetime64[ns]')
afk.info()

ValueError: time data "2024-09-04" doesn't match format "%Y%m%d", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
afk

,Tanggal,Data
0,2024-09-04 00:00:00.000000000,A
1,2024-03-07 00:00:00.000000000,B
2,1970-01-01 00:00:00.000202311,None


In [ ]:
afk['Datetime']= afk['Tanggal'].dt.to_period('D')
afk

,Tanggal,Data,Datetime
0,2024-09-04 00:00:00.000000000,A,2024-09-04
1,2024-03-07 00:00:00.000000000,B,2024-03-07
2,1970-01-01 00:00:00.000202311,None,1970-01-01


In [ ]:
from operator import attrgetter
afk['distance']= (afk['Datetime']-afk['Datetime']).apply(attrgetter('n'))+1
afk

,Tanggal,Data,Datetime,distance
0,2024-09-04 00:00:00.000000000,A,2024-09-04,1
1,2024-03-07 00:00:00.000000000,B,2024-03-07,1
2,1970-01-01 00:00:00.000202311,None,1970-01-01,1


In [ ]:
ok= (afk.loc[0,'Datetime']-afk.loc[0,'Datetime'])
print(dir(ok))

['__add__', '__array_priority__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__mul__', '__ne__', '__neg__', '__new__', '__pyx_vtable__', '__radd__', '__reduce__', '__reduce_cython__', '__reduce_ex__', '__repr__', '__rmul__', '__rsub__', '__rtruediv__', '__setattr__', '__setstate__', '__setstate_cython__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__truediv__', '_adjust_dst', '_apply', '_apply_array', '_as_pd_timedelta', '_attributes', '_cache', '_creso', '_day_opt', '_from_name', '_get_offset_day', '_nanos_inc', '_next_higher_resolution', '_offset_str', '_params', '_period_dtype_code', '_prefix', '_repr_attrs', '_use_relativedelta', '_validate_n', 'base', 'copy', 'delta', 'freqstr', 'is_anchored', 'is_month_end', 'is_month_start', 'is_on_offset', 'is_quarter_end', 'is_quarter_start', 'is_year_end', 'is_year_start', 'kwds

In [ ]:
bl= afk.dropna(how='any')
vl= afk[~afk['Data'].isna()]
bl

,Tanggal,Data,Datetime,distance
0,2024-09-04,A,2024-09-04,1
1,2024-03-07,B,2024-03-07,1


In [ ]:
a=['A3435','A786','B25246','78478']

pep=[]
for i in a:
    p=re.findall(r'\D',i)
    if p not in pep and p!=[]:
        pep.append(p)
print(pep)

[['A'], ['B']]


In [ ]:
bl[(bl['Data'].str.contains('A'))|(bl['Data'].str.contains('B'))]

,Tanggal,Data,Datetime,distance
0,2024-09-04,A,2024-09-04,1
1,2024-03-07,B,2024-03-07,1
